# Bill Breaker: train the model and put it online

This notebook trains the Bill Breaker model and hosts it for billbreaker.org.

1. Go to **Runtime > Change runtime type**, pick **T4 GPU**, and click **Save**.
2. Go to **Runtime > Run all**.
3. A box near the top asks for a **Hugging Face token**. To get one:
   - Sign in at huggingface.co (or make a free account).
   - Open **huggingface.co/settings/tokens** and click **Create new token**.
   - Choose **Write**, give it any name, then create and copy it.
   - Paste it into the box and press Enter.
4. Leave this tab open. Training takes about an hour on a T4.
5. The last cell prints a link. Set it as `ML_ENDPOINT` in the site's Vercel settings.

The trained model is saved to your Hugging Face account, so nothing is lost if Colab disconnects afterwards. If no GPU is free, the notebook still runs on CPU with a smaller, quicker training set.

## 1. Sign in to Hugging Face

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "transformers>=4.44", "datasets>=2.20", "accelerate>=0.33",
                       "sentencepiece", "huggingface_hub>=0.24"])

from getpass import getpass
from huggingface_hub import HfApi, login

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = getpass("Paste your Hugging Face token (Write access) and press Enter: ").strip()

login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
USER = api.whoami()["name"]
MODEL_REPO = f"{USER}/bill-breaker-t5"
SPACE_REPO = f"{USER}/billbreaker"
print("Signed in to Hugging Face as", USER)

## 2. Training data: BillSum (bill text paired with official summaries)

In [ ]:
import random
import numpy as np
import torch
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

GPU = torch.cuda.is_available()
N_TRAIN, EPOCHS = (2000, 2) if GPU else (300, 1)
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none, using CPU with a smaller training set")

billsum = load_dataset("FiscalNote/billsum", split="train")
pool = billsum.shuffle(seed=SEED).select(range(N_TRAIN))
splits = pool.train_test_split(test_size=0.1, seed=SEED)
train_ds, val_ds = splits["train"], splits["test"]
print("Training bills:", len(train_ds), "| validation bills:", len(val_ds))

## 3. Fine-tune FLAN-T5

In [ ]:
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)

BASE_MODEL = "google/flan-t5-base"
PROMPT = "Explain this bill in plain English: "
MAX_SOURCE, MAX_TARGET = 1024, 256

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)


def preprocess(batch):
    enc = tok([PROMPT + t for t in batch["text"]], max_length=MAX_SOURCE, truncation=True)
    enc["labels"] = tok(text_target=batch["summary"], max_length=MAX_TARGET, truncation=True)["input_ids"]
    return enc


cols = train_ds.column_names
train_tok = train_ds.map(preprocess, batched=True, remove_columns=cols)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=cols)

args = Seq2SeqTrainingArguments(
    output_dir="bill-breaker-t5",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    gradient_checkpointing=True,
    logging_steps=25,
    report_to="none",
    seed=SEED,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForSeq2Seq(tok, model=model),
)
trainer.train()
model = trainer.model.eval()
model.config.use_cache = True

## 4. Save the model to your Hugging Face account

In [ ]:
model.push_to_hub(MODEL_REPO, commit_message="Bill Breaker fine-tuned FLAN-T5")
tok.push_to_hub(MODEL_REPO, commit_message="Tokenizer")
print("Model saved to https://huggingface.co/" + MODEL_REPO)

## 5. Put the model online (a free Hugging Face Space)

In [ ]:
SPACE_FILES = {
 "app.py": "\"\"\"Bill Breaker model server (runs as a free Hugging Face Space).\n\nPOST /explain {\"text\", \"title\", \"profile\"} returns\n{\"general\", \"personalized\", \"sections\"}: the fine-tuned FLAN-T5 summary, a version\nrewritten for the reader by Qwen2.5-1.5B-Instruct, and the sections that drove it.\nReader answers are used for this one request only; nothing is stored or logged.\n\"\"\"\n\nimport os\nfrom functools import lru_cache\n\nimport torch\nfrom fastapi import FastAPI, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom pydantic import BaseModel, Field\nfrom transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline\n\nfrom sections import clean_profile, rank_sections\n\nMODEL_ID = os.environ.get(\"MODEL_ID\", \"\")\nPERSONALIZER = os.environ.get(\"PERSONALIZER\", \"Qwen/Qwen2.5-1.5B-Instruct\")\nPROMPT = \"Explain this bill in plain English: \"\nMAX_SOURCE, MAX_TARGET = 1024, 256\nALLOWED_ORIGINS = [\n    \"https://billbreaker.org\",\n    \"https://www.billbreaker.org\",\n    \"http://localhost:3000\",\n]\n\nSYSTEM = (\n    \"You explain US congressional bills to ordinary people in plain English at about an 8th-grade reading level. \"\n    \"Use only the bill summary and bill sections you are given; never invent facts, numbers, or effects. \"\n    \"A bill is a proposal and may never have become law, so do not say it is law. \"\n    \"Stay neutral: no political opinions and no advice on how to vote. \"\n    \"Write two short parts with these headings: 'What this bill does' (3 to 5 bullet points) and \"\n    \"'What it could mean for you' (2 to 4 sentences tied to the reader's details). \"\n    \"If the bill does not clearly affect this reader, say that plainly instead of stretching.\"\n)\n\ntorch.set_num_threads(os.cpu_count() or 2)\n\napp = FastAPI(title=\"Bill Breaker model\")\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=ALLOWED_ORIGINS,\n    allow_methods=[\"POST\", \"GET\"],\n    allow_headers=[\"content-type\"],\n)\n\n\nclass ExplainRequest(BaseModel):\n    text: str = Field(min_length=80, max_length=200_000)\n    title: str = Field(default=\"This bill\", max_length=300)\n    profile: dict[str, str] = Field(default_factory=dict)\n\n\n@lru_cache(maxsize=1)\ndef summarizer():\n    if not MODEL_ID:\n        raise RuntimeError(\"MODEL_ID is not set\")\n    tok = AutoTokenizer.from_pretrained(MODEL_ID)\n    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).eval()\n    return tok, model\n\n\n@lru_cache(maxsize=1)\ndef writer():\n    return pipeline(\"text-generation\", model=PERSONALIZER, torch_dtype=torch.float32, device_map=\"cpu\")\n\n\ndef summarize(text):\n    tok, model = summarizer()\n    inputs = tok(PROMPT + text, max_length=MAX_SOURCE, truncation=True, return_tensors=\"pt\")\n    with torch.no_grad():\n        out = model.generate(**inputs, max_new_tokens=MAX_TARGET, num_beams=4,\n                             no_repeat_ngram_size=3, early_stopping=True)\n    return tok.decode(out[0], skip_special_tokens=True)\n\n\ndef personalize(title, general, sections, profile):\n    excerpts = \"\\n\\n\".join(\n        f\"SEC. {s['section']} {s['heading']}\\n{' '.join(s['body'].split()[:250])}\" for s in sections\n    ) or \"(No sections matched this reader's details.)\"\n    reader = \"\\n\".join(f\"- {k}: {v}\" for k, v in profile.items()) or \"- (no details given)\"\n    user = (f\"Bill: {title}\\n\\nGeneral summary:\\n{general}\\n\\n\"\n            f\"Sections most related to the reader:\\n{excerpts}\\n\\nAbout the reader:\\n{reader}\")\n    out = writer()([{\"role\": \"system\", \"content\": SYSTEM}, {\"role\": \"user\", \"content\": user}],\n                   max_new_tokens=350, do_sample=False)\n    return out[0][\"generated_text\"][-1][\"content\"]\n\n\n@app.on_event(\"startup\")\ndef warm_up():\n    summarizer()\n    writer()\n\n\n@app.get(\"/\")\ndef health():\n    return {\"ok\": True, \"model\": MODEL_ID, \"personalizer\": PERSONALIZER}\n\n\n@app.post(\"/explain\")\ndef explain(req: ExplainRequest):\n    profile = clean_profile(req.profile)\n    try:\n        general = summarize(req.text)\n        sections = rank_sections(req.text, profile)\n        personalized = personalize(req.title, general, sections, profile)\n    except RuntimeError as err:\n        raise HTTPException(status_code=503, detail=str(err)) from err\n    return {\n        \"general\": general,\n        \"personalized\": personalized,\n        \"sections\": [\n            {k: s[k] for k in (\"section\", \"heading\", \"score\", \"matched_terms\")} for s in sections\n        ],\n    }\n",
 "sections.py": "\"\"\"Find the parts of a bill that matter to one reader.\n\nSame rules as section 8 of notebooks/bill_breaker.ipynb and billbreaker/ml on the site:\nsplit at SEC. headings, then score each section by how often the reader's terms appear.\n\"\"\"\n\nimport re\n\nSECTION_HEADING = re.compile(\n    r\"(?im)^\\s*(?:SEC(?:TION)?\\.?\\s+|\u00a7\\s*)(?P<no>\\d+[A-Z0-9]*(?:[-\u2013]\\d+[A-Z0-9]*)?)\"\n    r\"\\.?\\s*(?P<title>[^\\n]{0,200})\\s*$\"\n)\n\n# Answer word -> extra terms to look for in the bill text.\nEXPANSIONS = {\n    \"student\": [\"student\", \"college\", \"university\", \"tuition\", \"loan\", \"education\", \"school\"],\n    \"retired\": [\"retire\", \"Social Security\", \"pension\", \"Medicare\"],\n    \"65+\": [\"Medicare\", \"Social Security\", \"elderly\", \"older\", \"senior\"],\n    \"45-64\": [\"retire\", \"pension\"],\n    \"18-24\": [\"student\", \"young\", \"youth\"],\n    \"under 18\": [\"minor\", \"child\", \"youth\", \"school\"],\n    \"kids\": [\"child\", \"children\", \"family\", \"school\", \"parent\"],\n    \"elderly\": [\"elderly\", \"older\", \"senior\", \"caregiver\", \"Medicare\"],\n    \"medicare\": [\"Medicare\"],\n    \"medicaid\": [\"Medicaid\", \"low-income\"],\n    \"marketplace\": [\"health insurance\", \"exchange\", \"premium\"],\n    \"employer\": [\"employer\", \"group health plan\"],\n    \"va\": [\"veteran\", \"Department of Veterans Affairs\"],\n    \"veteran\": [\"veteran\", \"armed forces\", \"military\", \"Department of Veterans Affairs\"],\n    \"under $30k\": [\"low-income\", \"poverty\", \"assistance\", \"credit\"],\n    \"$30-75k\": [\"tax\", \"credit\", \"middle-income\"],\n    \"farm\": [\"agriculture\", \"farm\", \"crop\", \"rural\"],\n    \"teacher\": [\"teacher\", \"school\", \"education\"],\n    \"nurse\": [\"nurse\", \"hospital\", \"health care\"],\n    \"environment\": [\"environment\", \"conservation\", \"wildlife\", \"pollution\", \"climate\", \"wilderness\"],\n    \"taxes\": [\"tax\", \"Internal Revenue Code\", \"credit\", \"deduction\"],\n    \"public lands\": [\"public land\", \"national forest\", \"wilderness\", \"Bureau of Land Management\"],\n}\nMILITARY_TERMS = [\"veteran\", \"armed forces\", \"military\", \"service member\"]\nSTOPWORDS = set(\"just part time with work about most like public\".split())\nSKIPPED_ANSWERS = {\"no\", \"none\", \"skip\", \"n/a\", \"\"}\n\n\ndef clean_profile(profile):\n    \"\"\"Drop skipped answers and normalize dashes, as the notebook and site do.\"\"\"\n    out = {}\n    for key, value in (profile or {}).items():\n        text = re.sub(r\"[\u2013\u2014]\", \"-\", str(value or \"\")).strip()\n        if text.lower() not in SKIPPED_ANSWERS:\n            out[key] = text\n    return out\n\n\ndef split_bill_sections(bill_text, max_chunk_words=850):\n    bill_text = re.sub(r\"\\r\\n?\", \"\\n\", str(bill_text or \"\")).strip()\n    matches = list(SECTION_HEADING.finditer(bill_text))\n    if matches:\n        sections = []\n        for i, match in enumerate(matches):\n            end = matches[i + 1].start() if i + 1 < len(matches) else len(bill_text)\n            sections.append({\n                \"section\": match.group(\"no\"),\n                \"heading\": match.group(\"title\").strip(\" .-:\"),\n                \"body\": bill_text[match.end():end].strip(),\n            })\n        return sections\n    paragraphs = [p.strip() for p in re.split(r\"\\n{2,}\", bill_text) if p.strip()]\n    sections, carry, total = [], [], 0\n    for p in paragraphs:\n        n = len(p.split())\n        if carry and total + n > max_chunk_words:\n            sections.append({\"section\": str(len(sections) + 1), \"heading\": \"Unnumbered text chunk\",\n                             \"body\": \"\\n\\n\".join(carry)})\n            carry, total = [], 0\n        carry.append(p)\n        total += n\n    if carry:\n        sections.append({\"section\": str(len(sections) + 1), \"heading\": \"Unnumbered text chunk\",\n                         \"body\": \"\\n\\n\".join(carry)})\n    return sections\n\n\ndef profile_terms(profile):\n    terms = set()\n    for key, answer in profile.items():\n        low = answer.lower()\n        if key in (\"work\", \"interests\"):\n            terms.update(w for w in re.findall(r\"[a-z][a-z\\-]{3,}\", low) if w not in STOPWORDS)\n        if key == \"military\" and low.startswith(\"y\"):\n            terms.update(MILITARY_TERMS)\n        for trigger, extra in EXPANSIONS.items():\n            if re.search(r\"(?<![a-z])\" + re.escape(trigger) + r\"(?![a-z])\", low):\n                terms.update(e.lower() for e in extra)\n    if \"state\" in profile:\n        terms.add(profile[\"state\"].lower())\n    return sorted(terms)\n\n\ndef rank_sections(text, profile, top_k=3):\n    terms = profile_terms(profile)\n    rows = []\n    for s in split_bill_sections(text):\n        body = (s[\"heading\"] + \" \" + s[\"body\"]).lower()\n        hits = {t: len(re.findall(r\"\\b\" + re.escape(t), body)) for t in terms}\n        hits = {t: n for t, n in hits.items() if n}\n        rows.append({**s, \"score\": sum(hits.values()), \"matched_terms\": sorted(hits)})\n    ranked = sorted(rows, key=lambda r: -r[\"score\"])\n    return [r for r in ranked if r[\"score\"] > 0][:top_k]\n",
 "requirements.txt": "--extra-index-url https://download.pytorch.org/whl/cpu\ntorch==2.4.1\ntransformers==4.46.3\nsentencepiece==0.2.0\naccelerate==1.1.1\nfastapi==0.115.5\nuvicorn==0.32.1\n",
 "Dockerfile": "FROM python:3.11-slim\n\nRUN useradd -m -u 1000 user\nUSER user\nENV HOME=/home/user PATH=/home/user/.local/bin:$PATH HF_HOME=/home/user/.cache/huggingface\nWORKDIR /home/user/app\n\nCOPY --chown=user requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY --chown=user . .\n\nEXPOSE 7860\nCMD [\"uvicorn\", \"app:app\", \"--host\", \"0.0.0.0\", \"--port\", \"7860\"]\n",
 "README.md": "---\ntitle: Bill Breaker\ncolorFrom: gray\ncolorTo: yellow\nsdk: docker\napp_port: 7860\npinned: false\n---\n\nServes the Bill Breaker model for billbreaker.org. `POST /explain` with\n`{\"text\", \"title\", \"profile\"}` returns the fine-tuned FLAN-T5 summary, a version\nwritten for the reader, and the bill sections that drove it.\n\nSet by `notebooks/bill_breaker_deploy.ipynb`: the `MODEL_ID` variable points at the\nfine-tuned model on the Hugging Face Hub.\n"
}

In [ ]:
from pathlib import Path

space_dir = Path("billbreaker_space")
space_dir.mkdir(exist_ok=True)
for name, content in SPACE_FILES.items():
    (space_dir / name).write_text(content)

api.create_repo(SPACE_REPO, repo_type="space", space_sdk="docker", exist_ok=True)
api.add_space_variable(SPACE_REPO, "MODEL_ID", MODEL_REPO)
api.upload_folder(repo_id=SPACE_REPO, repo_type="space", folder_path=str(space_dir),
                  commit_message="Bill Breaker model server")
print("Space created: https://huggingface.co/spaces/" + SPACE_REPO)

## 6. Wait for it to start and test it

In [ ]:
import time
import requests

host = "https://" + SPACE_REPO.lower().replace("/", "-").replace("_", "-").replace(".", "-") + ".hf.space"
print("Waiting for the Space to build and load the models (usually 5 to 15 minutes)...")
deadline = time.time() + 40 * 60
while True:
    stage = api.get_space_runtime(SPACE_REPO).stage
    if stage in ("BUILD_ERROR", "RUNTIME_ERROR", "CONFIG_ERROR"):
        raise RuntimeError(f"The Space stopped with {stage}. Open https://huggingface.co/spaces/{SPACE_REPO} and check its Logs tab.")
    if stage == "RUNNING":
        try:
            if requests.get(host + "/", timeout=30).ok:
                break
        except requests.RequestException:
            pass
    if time.time() > deadline:
        raise TimeoutError(f"Still starting after 40 minutes. Check https://huggingface.co/spaces/{SPACE_REPO}")
    time.sleep(20)

sample = {
    "title": "Example bill",
    "text": "SEC. 1. SHORT TITLE. This Act may be cited as the Example Act. "
            "SEC. 2. STUDENT LOANS. The Secretary of Education shall reduce interest on "
            "federal student loans for borrowers enrolled at a college or university.",
    "profile": {"age": "18-24", "work": "college student"},
}
reply = requests.post(host + "/explain", json=sample, timeout=900)
reply.raise_for_status()
print(reply.json()["general"])
print()
print("=" * 60)
print("Done. Set this as ML_ENDPOINT on Vercel:")
print(host + "/explain")
print("=" * 60)